In [1]:
# import rpy2.robjects as ro

# # Source the R script
# ro.r['source']('downsample.R')

# # Get R function into Python
# downsample_clonotypes = ro.globalenv['downsample_clonotypes']

# # Call the function with your parameters
# downsample_clonotypes(num_clonotypes=10000, threshold=10000, prob=True)

In [2]:
import chephy_model as cpm
import gnn_utils
import torch
import pandas as pd


/home/dsi/chenbis/anaconda3/envs/chephy2/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Using cache found in /home/dsi/chenbis/.cache/torch/hub/facebookresearch_esm_main


In [3]:
atchley_path = "/home/dsi/chenbis/repos/sol_lab/files/atchley.csv"
atchley_df = pd.read_csv(atchley_path)
atchley_dict = atchley_df.set_index("amino.acid").to_dict(orient="index")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

downsample_size = 600
chain = "AB"

downsampled_path = f"/dsi/efroni-lab/sbm/chen2/downsampled_data/ovarian/{chain}/{downsample_size}/"

graph_path = f"/dsi/efroni-lab/sbm/chen2/graph_data/ovarian/{chain}/{downsample_size}"

In [4]:
from pathlib import Path
file_names = [downsampled_path + f.name for f in Path(downsampled_path).iterdir() if f.is_file()]

In [5]:
def prepare_data(data, cdr3_header):

    # remove cdr3 sequences that contain non-aa letters (this also removes nan)
    amino_acid_pattern = r'^[ARNDCEQGHILKMFPSTWYV]+$'

    data_filtered = data[data[cdr3_header].str.match(amino_acid_pattern, case=False, na=False)]

    data_filtered = data_filtered[data_filtered[cdr3_header].str.len() >= 8]

    return data_filtered

In [6]:
from scipy.spatial.distance import squareform 

def get_graph_data_full_nodes(
    data, cdr3_header, label, distance_threshold=0.2, use_mst=True, enrich=False
):
    data = prepare_data(data, cdr3_header)

    # unique full sequences = nodes
    full_nodes = sorted(set(data[cdr3_header].dropna()))
    full_idx = {s: i for i, s in enumerate(full_nodes)}
    data["tcr_index"] = data[cdr3_header].map(full_idx)

    # build a mapping full_seq -> truncated_seq for distance computation
    tmp = pd.DataFrame({cdr3_header: full_nodes})
    tmp = cpm.truncate_sequences(tmp, cdr3_header, 4, 4)   # adds cdr3_truncated
    trunc_list = tmp["cdr3_truncated"].tolist()

    # compute distances between *truncated* strings, aligned to full_nodes order
    # easiest: feed the set, then reindex back (because your cpm returns its own order)
    trunc_set = set(trunc_list)
    chephy_matrix, trunc_order = cpm.find_che_phy_dist(trunc_set)
    trunc_idx = {s: i for i, s in enumerate(trunc_order)}

    # build full square distance matrix between nodes (full_nodes order)
    D_trunc = squareform(chephy_matrix)
    import numpy as np
    D = np.zeros((len(full_nodes), len(full_nodes)), dtype=float)
    for i, ti in enumerate(trunc_list):
        for j, tj in enumerate(trunc_list):
            D[i, j] = D_trunc[trunc_idx[ti], trunc_idx[tj]]

    # node weights (sum readFraction per full seq)
    node_weights = (
        data.dropna(subset=["tcr_index"])
            .groupby("tcr_index")["readFraction"].sum()
            .reindex(range(len(full_nodes)))
            .fillna(0.0)
            .to_numpy()
            .tolist()
    )

    graph = gnn_utils.build_graph(
        D,
        full_nodes,  # full sequences as node feature
        label=label,
        node_weights=node_weights,
        distance_threshold=distance_threshold,
        use_mst=use_mst,
        enrich=enrich
    )
    return graph, data


In [8]:
from tqdm import tqdm
import os

cdr3_header = "aaSeqCDR3"

distance_threshold=0.08
use_mst=True
enrich=True

sub_folder = f"{graph_path}/{distance_threshold}_{use_mst}_{enrich}"

if not os.path.exists(sub_folder):
    os.makedirs(sub_folder)


for file_name in tqdm(file_names, desc="Processing files"):
    df = pd.read_csv(file_name, sep="\t")
    label = file_name.split("/")[-1].split("_")[0]
    file_save_name = f"{sub_folder}/{file_name.rsplit('/', 1)[-1]}.pt"

    if label == "sick":
        label = 1
    else:
        label = 0
        
    data = get_graph_data_full_nodes(df, cdr3_header, label=label, distance_threshold=distance_threshold, use_mst=use_mst, enrich=enrich)

    torch.save(data, file_save_name)


Processing files: 100%|██████████| 84/84 [09:28<00:00,  6.77s/it]  
